# Resume Classifier – Machine Learning Project

**Goal:** Predict the job category of a resume using its text.

Dataset expected: `Resume.csv` with columns `ID`, `Resume_str`, `Resume_html`, and `Category`.

**Workflow:** Load data → inspect and clean → train/test split → TF-IDF → Logistic Regression → evaluate → save model → predict a new resume.

> Run the cells from top to bottom in Google Colab. This notebook uses `Resume_str` as input and `Category` as the target.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

## 2. Upload the dataset to Google Colab

In [ ]:
from google.colab import files

uploaded = files.upload()  # Select Resume.csv from your computer

In [ ]:
# Read the uploaded CSV file
df = pd.read_csv("Resume.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

## 3. Understand the dataset

In [ ]:
print("Number of rows and columns:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

print("\nNumber of resume categories:", df["Category"].nunique())
print("\nCategory counts:")
print(df["Category"].value_counts())

In [ ]:
# Visualize the number of resumes in each category
plt.figure(figsize=(12, 7))
df["Category"].value_counts().plot(kind="bar")
plt.title("Number of Resumes in Each Category")
plt.xlabel("Resume Category")
plt.ylabel("Number of Resumes")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

## 4. Select input (X) and target (y)
- **X:** Resume text (`Resume_str`)
- **y:** Resume category (`Category`)

In [ ]:
# Keep only the columns required for this project
df = df[["Resume_str", "Category"]].copy()

# Handle any missing values safely
df = df.dropna(subset=["Resume_str", "Category"])

# Convert text and labels to strings and remove extra whitespace
df["Resume_str"] = df["Resume_str"].astype(str).str.strip()
df["Category"] = df["Category"].astype(str).str.strip()

# Remove empty resume text
df = df[df["Resume_str"] != ""]

X = df["Resume_str"]
y = df["Category"]

print("Input feature:", X.name)
print("Target variable:", y.name)
print("Total resumes:", len(df))
print("Total categories:", y.nunique())

## 5. Split data into training and testing sets
We use 80% of the data for training and 20% for testing. `stratify=y` helps preserve category proportions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

## 6. Convert resume text into numbers using TF-IDF
TF-IDF means **Term Frequency–Inverse Document Frequency**. It converts text into numerical features based on word frequency and importance.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

# Learn vocabulary and IDF weights from training data only
X_train_tfidf = vectorizer.fit_transform(X_train)

# Use the same learned vocabulary to transform test data
X_test_tfidf = vectorizer.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

## 7. Train the Logistic Regression model

In [ ]:
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

model.fit(X_train_tfidf, y_train)

print("Model training completed!")

## 8. Make predictions on test data

In [ ]:
y_pred = model.predict(X_test_tfidf)

print("First 10 actual categories:")
print(y_test.head(10).tolist())

print("\nFirst 10 predicted categories:")
print(y_pred[:10].tolist())

## 9. Evaluate model performance

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 10. Confusion matrix

In [ ]:
labels = model.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(16, 13))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)
plt.title("Resume Classifier – Confusion Matrix")
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Save the trained model and TF-IDF vectorizer
These files can later be used in a Streamlit application.

In [ ]:
joblib.dump(model, "resume_classifier_model.pkl")
joblib.dump(vectorizer, "resume_tfidf_vectorizer.pkl")

print("Saved: resume_classifier_model.pkl")
print("Saved: resume_tfidf_vectorizer.pkl")

In [ ]:
# Download the saved files from Colab
from google.colab import files

files.download("resume_classifier_model.pkl")
files.download("resume_tfidf_vectorizer.pkl")

## 12. Predict the category of a new resume
Replace the sample text with the text from a resume you want to classify.

In [ ]:
new_resume = """
Python developer with experience in Python, pandas, NumPy, machine learning,
SQL, data analysis, and building web applications.
"""

# Convert new resume text using the already-fitted TF-IDF vectorizer
new_resume_tfidf = vectorizer.transform([new_resume])

# Predict the category
predicted_category = model.predict(new_resume_tfidf)[0]

print("Predicted Resume Category:", predicted_category)

## 13. Important notes

1. This is a **24-class classification** project: the model predicts one category from the categories present in the dataset.
2. The notebook uses `Resume_str` as the text input and `Category` as the target. `ID` and `Resume_html` are not used.
3. TF-IDF is fitted only on training data to avoid data leakage.
4. The dataset has fewer examples in some categories, so review the per-category precision, recall and F1-score—not accuracy alone.
5. A predicted category is a model estimate, not a hiring decision. Review resumes fairly and keep a human involved.